# Assignment 2 : Language Modeling, BM25 & Evaluation

**Student names**: Brage Andreas Hoven <br>
**Group number**: 99 <br>
**Date**: 2026-10-02

## Important notes
Please read and follow these rules. Submissions that do not fulfill them may be returned.
1. You may work in groups of maximum 2 students.
2. Submit in **.ipynb** format only.
3. The assignment must be typed. Handwritten answers are not accepted.

**On libraries.** The TODO cells are yours to implement. numpy, pandas, scipy.sparse and matplotlib are fine underneath. Do not use libraries that implement a step for you, such as scikit-learn vectorizers or metrics, NLTK tokenizers, or IR evaluation packages.

**Due date**: 02.10.2026 23:59

### What you will do
- Build a **unigram document language model** with a **document-term matrix**.
- Rank documents for queries using **Jelinek-Mercer smoothing**.
- (Optional) Try **Dirichlet** and compare briefly.
- Rank the same queries using **BM25**.
- Evaluate and compare the two models using **Cranfield queries and qrels** (P@k, MAP, MRR).

---
## Dataset

Make sure the Cranfield files are placed next to the notebook:
- `cran.all.1400` — document collection (1400 docs)
- `cran.qry` — queries
- `cranqrel` — relevance judgments (qrels)

> Only the **document parsing** for cran.all.1400 code is provided below. You will implement the rest in the TODO cells.


### Load and parse documents (provided)

Run the cell to parse the Cranfield documents. Update the path so it points to your `cran.all.1400` file.


In [ ]:
# Read 'cran.all.1400' and parse the documents into a suitable data structure

CRAN_PATH = r"your_path_to/cran.all.1400"  # <-- change this!

def parse_cranfield(path):
    docs = {}
    current_id = None
    current_field = None
    buffers = {"T": [], "A": [], "B": [], "W": []}
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith(".I "):
                if current_id is not None:
                    docs[current_id] = {
                        "id": current_id,
                        "title": " ".join(buffers["T"]).strip(),
                        "abstract": " ".join(buffers["W"]).strip()
                    }
                current_id = int(line.split()[1])
                buffers = {k: [] for k in buffers}
                current_field = None
            elif line.startswith("."):
                tag = line[1:].strip()
                current_field = tag if tag in buffers else None
            else:
                if current_field is not None:
                    buffers[current_field].append(line)
    if current_id is not None:
        docs[current_id] = {
            "id": current_id,
            "title": " ".join(buffers["T"]).strip(),
            "abstract": " ".join(buffers["W"]).strip()
        }
    print(f"Parsed {len(docs)} documents.")
    return docs

docs = parse_cranfield(CRAN_PATH)
print(list(docs.items())[:1])  # peek at the first parsed doc

## 2.1 Ranking Models

You will create a **unigram language model** per document, using a **document-term matrix**, and score queries with **Jelinek-Mercer smoothing**. You will then rank the same queries with **BM25**, using the same matrix.

### 2.1.1 Preprocessing

Implement a simple tokenizer/normalizer (e.g., lowercasing, punctuation removal and stopword removal) and apply it to each document

- Return a list of tokens for each document.


In [ ]:
# TODO: Implement preprocessing and apply to all documents

STOPWORDS = set("""a about above after again against all am an and any are aren't as at be because been
before being below between both but by can't cannot could couldn't did didn't do does doesn't doing don't down
during each few for from further had hadn't has hasn't have haven't having he he'd he'll he's her here here's hers
herself him himself his how how's i i'd i'll i'm i've if in into is isn't it it's its itself let's me more most
mustn't my myself no nor not of off on once only or other ought our ours ourselves out over own same shan't she
she'd she'll she's should shouldn't so some such than that that's the their theirs them themselves then there there's
these they they'd they'll they're they've this those through to too under until up very was wasn't we we'd we'll we're
we've were weren't what what's when when's where where's which while who who's whom why why's with won't would wouldn't
you you'd you'll you're you've your yours yourself yourselves""".split())

# Your code here

### 2.1.2 Build the matrix

Construct:
- A vocabulary `term -> column_index`
- A (sparse) **document–term count matrix**
- Document lengths `|d|` and collection (all documents) totals
- Document frequencies `n_t` (number of documents containing the term) and the average document length (needed for BM25)

> Tip: You may use dictionaries or `scipy.sparse` for efficiency.

In [ ]:
# TODO: Build vocabulary, doc-term counts (sparse), document lengths, collection totals,
# document frequencies, and average document length
# Your code here

### 2.1.3 Rank with **Jelinek-Mercer smoothing**

Implement query likelihood scoring with Jelinek-Mercer smoothing:

$\hat{P}(t \mid M_d) = \lambda \hat{P}_{\text{mle}}(t \mid M_d) + (1 - \lambda)\hat{P}_{\text{mle}}(t \mid M_c), \ \lambda = 0.5$



In [ ]:
""" TODO: Create a function for implementing query likelihood scoring with Jelinek Mercer smoothing (λ=0.5), 
using the formula above."""

def jelinek_smoothing(query: str):
    # Your code here
    raise NotImplementedError("Implement Jelinek-Mercer smoothing here")

In [ ]:
# Do not change this code
queries_assignment2 = [
  "gas pressure",
  "structural aeroelastic flight high speed aircraft",
  "heat conduction composite slabs",
  "boundary layer control",
  "compressible flow nozzle",
  "combustion chamber injection",
  "laminar turbulent transition",
  "fatigue crack growth",
  "wing tip vortices",
  "propulsion efficiency"
]

In [ ]:
# Run Jelinek-Mercer smoothing on queries in batch (print top-10 results for each), using the function you created
def run_batch_smoothing(queries):
    results = {}
    for i, q in enumerate(queries, 1):
        res = jelinek_smoothing(q)
        results[f"Q{i}"] = res
    return results

jelinek_results = run_batch_smoothing(queries_assignment2)

for qid, res in jelinek_results.items():
    print(qid, "=>", res[:10])


#### Dirichlet

If you have time, also implement Dirichlet smoothing and briefly compare the top-10 lists for the first query in the queries_assignment2 list


In [ ]:
# TODO (Optional): Implement Dirichlet scoring and compare with Jelinek-Mercer
# Your code here


### 2.1.4 Rank with **BM25**

Implement BM25 ranking, using the formulas below:

$B_{i,j} = \dfrac{(K_1 + 1) \cdot tf_{i,j}}{K_1 \cdot \left[(1 - b) + b \cdot \dfrac{len(d_i)}{avg\_doclen}\right] + tf_{i,j}}$

$sim_{BM25}(d_i, q) \sim \sum\limits_{w_j \in q \,\wedge\, w_j \in d_i} B_{i,j} \cdot \log\left(\dfrac{N - n_j + 0.5}{n_j + 0.5}\right)$

where $tf_{i,j}$ is the frequency of term $w_j$ in document $d_i$, $len(d_i)$ is the document length, $avg\_doclen$ is the average document length in the collection, $N$ is the number of documents and $n_j$ is the number of documents containing $w_j$. Use $K_1 = 1$ and $b = 0.75$.

In [ ]:
""" TODO: Create a function for implementing BM25 ranking (K1=1, b=0.75),
using the formulas above."""

def bm25_ranking(query: str):
    # Your code here
    raise NotImplementedError("Implement BM25 ranking here")

In [ ]:
# Run BM25 on queries in batch (print top-10 results for each), using the function you created
def run_batch_bm25(queries):
    results = {}
    for i, q in enumerate(queries, 1):
        res = bm25_ranking(q)
        results[f"Q{i}"] = res
    return results

bm25_results = run_batch_bm25(queries_assignment2)

for qid, res in bm25_results.items():
    print(qid, "=>", res[:10])

## 2.2 Evaluation (Cranfield queries + qrels)

Evaluate your two ranking models using **Cranfield**:
- Parse **queries** from `cran.qry`
- Parse **relevance judgments** from `cranqrel`
- Compute **P@k (k=5,10)**, **MAP (Mean Average Precision)**, and **MRR (Mean Reciprocal Rank)** over all queries

### 2.2.1 Parse `cran.qry` and `cranqrel`

- Create `queries[qid] = "text"` by parsing `cran.qry`, where `qid` is the position of the query in the file (1, 2, ..., 225), not the number after `.I`
- Create `qrels[qid] = set(relevant_doc_ids)` by parsing `cranqrel`, with the IDs as integers

The `.I` numbers in `cran.qry` have gaps (001, 002, 004, 008, ...), but `cranqrel` refers to queries by position. For example, the third query in the file is `.I 004`, and its judgments are listed under query 3.

The third column of `cranqrel` is a relevance code. Treat codes 1 to 4 as relevant, and ignore lines with code `-1`. Documents not listed for a query are not relevant.

Run the check cell below your code to confirm the parsing is correct.

In [ ]:
# TODO: Parse cran.qry and cranqrel into convenient data structures
# Your code here


In [ ]:
# Check your parsing. Do not change this code
assert sorted(queries) == list(range(1, 226)), "Number the queries 1 to 225 by their position in cran.qry"
assert "conduction" in str(queries[3]).lower(), "Query 3 should be about heat conduction in composite slabs"
assert 184 in qrels[1] and 486 not in qrels[1], "Check qrels: IDs should be integers, and lines with code -1 ignored"
print("Parsing looks correct")

### 2.2.2 Implement metrics: P@k, MAP, MRR

Write functions to compute:
- Precision@k (for **k=5** and **k=10**)
- Mean Average Precision (MAP)
- Mean Reciprocal Rank (MRR)


In [ ]:
# TODO: Implement P@k (k=5,10), MAP, and MRR
# Your code here


### 2.2.3 Evaluate your runs

- For **all queries**, generate rankings with your **Jelinek-Mercer** model and with your **BM25** model
- Report aggregate metrics for both models: P@5, P@10, MAP, MRR

In [ ]:
# TODO: Run retrieval for all queries with both models and compute P@5, P@10, MAP, and MRR for each
# Your code here

### 2.2.4 Interpolated precision–recall curves (11‑point)

- Using your **rankings from task 2.2.3** and the **relevance judgments (`cranqrel`)**, compute **precision** and **recall** at each rank for each query.
- For the 11 standard recall levels `R = {0.0, 0.1, ..., 1.0}`, compute the **interpolated precision** at level `r` as the **maximum precision** observed at any point with recall ≥ `r`.
- **Report/plot** the **11‑point interpolated precision–recall curve** across queries for **both models** (Jelinek-Mercer and BM25) in the same figure (and optionally a few per‑query curves).

In [ ]:
# TODO: Compute the 11-point interpolated precision–recall for each query, for both models

# Your code here